# Multimodal RAG Benchmarking with MMDocRAG

Benchmark: **MMDocRAG**

Kuicai Dong, Yujing Chang, Shĳie Huang, Yasheng Wang, Ruiming Tang, & Yong Liu. (2025). Benchmarking Retrieval-Augmented Multimomal Generation for Document Question Answering.

(Note: processing logs have been removed due to excessive length)

## Dataset Download

In [4]:
!git clone https://huggingface.co/datasets/MMDocIR/MMDocRAG

Cloning into 'MMDocRAG'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 36 (delta 8), reused 0 (delta 0), pack-reused 4 (from 1)
Receiving objects: 100% (36/36), 14.25 KiB | 7.12 MiB/s, done.
Resolving deltas: 100% (8/8), done.
Filtering content: 100% (7/7), 3.22 GiB | 9.95 MiB/s, done.


In [1]:
!unzip MMDocRAG/doc_pdfs.zip -d MMDocRAG/
!unzip MMDocRAG/images.zip -d MMDocRAG/

Archive:  MMDocRAG/doc_pdfs.zip
   creating: MMDocRAG/doc_pdfs/
  inflating: MMDocRAG/doc_pdfs/05-03-18-political-release.pdf  
  inflating: MMDocRAG/doc_pdfs/0b85477387a9d0cc33fca0f4becaa0e5.pdf  
  inflating: MMDocRAG/doc_pdfs/0e94b4197b10096b1f4c699701570fbf.pdf  
  inflating: MMDocRAG/doc_pdfs/11-21-16-Updated-Post-Election-Release.pdf  
  inflating: MMDocRAG/doc_pdfs/12-15-15-ISIS-and-terrorism-release-final.pdf  
  inflating: MMDocRAG/doc_pdfs/2005.12872v3.pdf  
  inflating: MMDocRAG/doc_pdfs/2019713402.pdf  
  inflating: MMDocRAG/doc_pdfs/2020.acl-main.207.pdf  
  inflating: MMDocRAG/doc_pdfs/2020.acl-main.408.pdf  
  inflating: MMDocRAG/doc_pdfs/2020.acl-main.423.pdf  
  inflating: MMDocRAG/doc_pdfs/2020.acl-main.45.pdf  
  inflating: MMDocRAG/doc_pdfs/2020.acl-main.48.pdf  
  inflating: MMDocRAG/doc_pdfs/2020.acl-main.653.pdf  
  inflating: MMDocRAG/doc_pdfs/2020.emnlp-main.213.pdf  
  inflating: MMDocRAG/doc_pdfs/2020.findings-emnlp.139.pdf  
  inflating: MMDocRAG/doc_pdfs/20

## Imports

In [1]:
from config.settings import RAGConfig
from RAGPipeline import RAGSystem
from utils.benchmark_eval_helpers import load_json, output_to_file

2026-04-09 10:34:57.277915: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-09 10:34:58.515417: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-09 10:35:05.090831: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/javan/anaconda3/lib/python3.13/site-packages/clip/clip.py:6: UserWarning: pkg_resourc

## Dataset Load

In [2]:
dataset = load_json("./MMDocRAG/evaluation_15.jsonl")
dataset[0]

{'q_id': 0,
 'doc_name': '12-15-15-ISIS-and-terrorism-release-final',
 'domain': 'Research report / Introduction',
 'question': 'In 2015, how many percentage of surveyed adults, Republicans, and Democrats believed that the goverment was doing very/fairly well in reducing the threat of terrorism? Please write the answer in list format, e.g., ["3","2"]',
 'evidence_modality_type': ['chart', 'text'],
 'question_type': 'Descriptive',
 'text_quotes': [{'quote_id': 'text1',
   'type': 'text',
   'text': 'Terrorism has reshaped the public’s agenda, both at home and abroad. Currently,   $29\\%$   cite  terrorism   $(18\\%)$  , national security   $(8\\%)$   or ISIS   $(7\\%)$   as the most important problem facing the  country today. One year ago, just  $4\\%$   of the public cited any of these issues. And while ISIS already  ranked high among leading international dangers,   $83\\%$   now regard ISIS as a major threat to the  well-being of the U.S., up from  $67\\%$   in August 2014.  ',
   '

## RAG Pipeline

### Initialization

In [3]:
config = RAGConfig()

rag_system = RAGSystem(config)

Image store initialized at: ./image_store


/home/javan/anaconda3/lib/python3.13/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/javan/anaconda3/lib/python3.13/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type a

Multi-modal embedding model (DINOv2+Talk2DINO) loaded.
Multi-vector database loaded.


### Ingest and Process Documents

In [ ]:
rag_system.ingest_documents("./MMDocRAG/doc_pdfs")

### Generate Answers

In [4]:
import time
from typing import List, Dict, Optional
from tqdm import tqdm
from datetime import datetime
import json
import os

In [16]:
def generate_benchmark_answers_sync(
        rag_system: RAGSystem,
        dataset: List[Dict],
        output_path: str = "./MMDocRAG/results.jsonl",
        rate_limit_delay: float = 0.5,
        resume: bool = True
    ) -> List[Dict]:
    """
    Generates answers for all questions in the benchmark dataset.
    
    args:
    - rag_system (RAGSystem): Initialized RAGSystem instance
    - dataset (List[Dict]): a list of benchmark items
    - output_path (str): path to save results
    - rate_limit_delay (float): seconds to wait between LLM calls
    - resume (bool): whether to skip already processed items from output file

    returns:
    - a list of result dictionaries with answers
    """
    processed_questions = set()
    results = []
    
    if resume and os.path.exists(output_path):
        try:
            with open(output_path, 'r') as f:
                for line in f:
                    item = json.loads(line)
                    processed_questions.add(item.get('question', ''))
                    results.append(item)
            print(f"Resumed from {len(results)} previously processed questions\n")
        except Exception as e:
            print(f"Could not resume: {e}. Starting fresh.\n")
    
    total = len(dataset)
    
    with tqdm(total=total, desc="Generating answers", initial=len(results)) as pbar:
        for idx, item in enumerate(dataset):
            # skip if already processed
            if item.get('question') in processed_questions:
                pbar.update(1)
                continue
            
            try:
                question = item.get('question')
                qid = item.get('qid', idx)
                print(f"Question: {question}")
                
                rag_result = rag_system.query(question)
                
                result = {
                    'qid': qid,
                    'question': question,
                    'answer': rag_result.get('answer', ''),
                    'timestamp': datetime.now().isoformat(),
                    'metadata': {
                        'retrieval_mode': rag_result.get('retrieval_mode'),
                        'documents_retrieved': rag_result.get('retrieval_metadata', {}).get('documents_retrieved', 0),
                        'images_used': rag_result.get('generation_metadata', {}).get('images_used', 0),
                    }
                }
                
                if 'answer_short' in item:
                    result['reference_answer_short'] = item['answer_short']

                if 'answer_interleaved' in item:
                    result['reference_answers'] = item['answer_interleaved']
                    
                if 'text_quotes' in item:
                    result['reference_documents'] = item['text_quotes']
                
                with open(output_path, 'a') as f:
                    json.dump(result, f)
                    f.write('\n')
                
                results.append(result)
                
            except Exception as e:
                print(f"\nError at item {idx}: {str(e)}")
            
            # rate limiting
            time.sleep(rate_limit_delay)
            pbar.update(1)
    
    print(f"\nCompleted! Total processed: {len(results)}")
    
    return results

In [6]:
from utils.benchmark_eval_helpers import grade_with_llm_judge

### Multimodal Retrieval

GPT-4o, with multimodal retrieval; $k$=3

In [ ]:
results = generate_benchmark_answers_sync(
    rag_system=rag_system,
    dataset=dataset,
    output_path="./MMDocRAG/results.jsonl",
    rate_limit_delay=1.0,
    resume=True
)

In [22]:
grading_data = []
for result in results:
    item = {
        'id': result.get('qid', ''),
        'question': result['question'],
        'llm_response': result['answer'],               # generated answer
        'answers': result.get('reference_answers', [])  # ground truth 
    }
    grading_data.append(item)

grading_results = grade_with_llm_judge(
    responses=grading_data,
    client=rag_system.generator.llm,
    output_file="./MMDocRAG/grading_results.json"
)

# results summary
print(f"\n{'='*50}")
print(f"Grading Summary")
print(f"{'='*50}")
print(f"Accuracy: {grading_results['accuracy']:.2%}")
print(f"Correct: {grading_results['correct_count']}/{grading_results['total_count']}")
print(f"\nDetailed results saved to: ./MMDocRAG/grading_results.json")


Grading 2000 generated responses using LLM judge...


Grading: 100%|██████████| 2000/2000 [1:08:24<00:00,  2.05s/it]


Detailed results saved to ./MMDocRAG/grading_results.json

Grading Summary
Accuracy: 49.30%
Correct: 986/2000

Detailed results saved to: ./MMDocRAG/grading_results.json


GPT-4o with multimodal retrieval, $k$=15

In [17]:
rag_system.config.top_k = 15

In [ ]:
results_k15 = generate_benchmark_answers_sync(
    rag_system=rag_system,
    dataset=dataset,
    output_path="./MMDocRAG/results_k15.jsonl",
    rate_limit_delay=1.0,
    resume=True
)

In [ ]:
grading_data_k15 = []
for result in results_k15:
    item = {
        'id': result.get('qid', ''),
        'question': result['question'],
        'llm_response': result['answer'],               # generated answer
        'answers': result.get('reference_answers', [])  # ground truth 
    }
    grading_data_k15.append(item)

grading_results_k15 = grade_with_llm_judge(
    responses=grading_data_k15,
    client=rag_system.generator.llm,
    output_file="./MMDocRAG/grading_results_k15.json"
)

# results summary
print(f"\n{'='*50}")
print(f"Grading Summary")
print(f"{'='*50}")
print(f"Accuracy: {grading_results_k15['accuracy']:.2%}")
print(f"Correct: {grading_results_k15['correct_count']}/{grading_results_k15['total_count']}")
print(f"\nDetailed results saved to: ./MMDocRAG/grading_results_k15.json")


Grading 2000 generated responses using LLM judge...


Grading: 100%|██████████| 2000/2000 [1:09:24<00:00,  2.08s/it]


Detailed results saved to ./MMDocRAG/grading_results_k15.json

Grading Summary
Accuracy: 57.50%
Correct: 1150/2000

Detailed results saved to: ./MMDocRAG/grading_results_k15.json


GPT-4o with multimodal retrieval, $k$=20

In [23]:
rag_system.config.top_k = 20

In [ ]:
results_k20 = generate_benchmark_answers_sync(
    rag_system=rag_system,
    dataset=dataset,
    output_path="./MMDocRAG/results_k20.jsonl",
    rate_limit_delay=1.0,
    resume=True
)

In [25]:
grading_data_k20 = []
for result in results_k20:
    item = {
        'id': result.get('qid', ''),
        'question': result['question'],
        'llm_response': result['answer'],               # generated answer
        'answers': result.get('reference_answers', [])  # ground truth 
    }
    grading_data_k20.append(item)

grading_results_k20 = grade_with_llm_judge(
    responses=grading_data_k20,
    client=rag_system.generator.llm,
    output_file="./MMDocRAG/grading_results_k20.json"
)

# results summary
print(f"\n{'='*50}")
print(f"Grading Summary")
print(f"{'='*50}")
print(f"Accuracy: {grading_results_k20['accuracy']:.2%}")
print(f"Correct: {grading_results_k20['correct_count']}/{grading_results_k20['total_count']}")
print(f"\nDetailed results saved to: ./MMDocRAG/grading_results_k20.json")


Grading 2000 generated responses using LLM judge...


Grading: 100%|██████████| 2000/2000 [1:08:39<00:00,  2.06s/it]


Detailed results saved to ./MMDocRAG/grading_results_k20.json

Grading Summary
Accuracy: 55.50%
Correct: 1110/2000

Detailed results saved to: ./MMDocRAG/grading_results_k20.json


GPT-4o with multimodal retrieval, $k$=5

In [ ]:
rag_system.config.top_k = 5

In [ ]:
results_k5 = generate_benchmark_answers_sync(
    rag_system=rag_system,
    dataset=dataset,
    output_path="./MMDocRAG/results_k5.jsonl",
    rate_limit_delay=1.0,
    resume=True
)

In [52]:
grading_data_k5 = []
for result in results_k5:
    item = {
        'id': result.get('qid', ''),
        'question': result['question'],
        'llm_response': result['answer'],               # generated answer
        'answers': result.get('reference_answers', [])  # ground truth 
    }
    grading_data_k5.append(item)

grading_results_k5 = grade_with_llm_judge(
    responses=grading_data_k5,
    client=rag_system.generator.llm,
    output_file="./MMDocRAG/grading_results_k5.json"
)

# results summary
print(f"\n{'='*50}")
print(f"Grading Summary")
print(f"{'='*50}")
print(f"Accuracy: {grading_results_k5['accuracy']:.2%}")
print(f"Correct: {grading_results_k5['correct_count']}/{grading_results_k5['total_count']}")
print(f"\nDetailed results saved to: ./MMDocRAG/grading_results_k5.json")


Grading 2000 generated responses using LLM judge...


Grading: 100%|██████████| 2000/2000 [1:07:52<00:00,  2.04s/it]


Detailed results saved to ./MMDocRAG/grading_results_k5.json

Grading Summary
Accuracy: 52.65%
Correct: 1053/2000

Detailed results saved to: ./MMDocRAG/grading_results_k5.json


GPT-4o with multimodal retrieval, $k$=7

In [59]:
rag_system.config.top_k = 7

In [ ]:
results_k7 = generate_benchmark_answers_sync(
    rag_system=rag_system,
    dataset=dataset,
    output_path="./MMDocRAG/results_k7.jsonl",
    rate_limit_delay=1.0,
    resume=True
)

In [61]:
grading_data_k7 = []
for result in results_k7:
    item = {
        'id': result.get('qid', ''),
        'question': result['question'],
        'llm_response': result['answer'],               # generated answer
        'answers': result.get('reference_answers', [])  # ground truth 
    }
    grading_data_k7.append(item)

grading_results_k7 = grade_with_llm_judge(
    responses=grading_data_k7,
    client=rag_system.generator.llm,
    output_file="./MMDocRAG/grading_results_k7.json"
)

# results summary
print(f"\n{'='*50}")
print(f"Grading Summary")
print(f"{'='*50}")
print(f"Accuracy: {grading_results_k7['accuracy']:.2%}")
print(f"Correct: {grading_results_k7['correct_count']}/{grading_results_k7['total_count']}")
print(f"\nDetailed results saved to: ./MMDocRAG/grading_results_k7.json")


Grading 2000 generated responses using LLM judge...


Grading: 100%|██████████| 2000/2000 [1:09:29<00:00,  2.08s/it]


Detailed results saved to ./MMDocRAG/grading_results_k7.json

Grading Summary
Accuracy: 55.55%
Correct: 1111/2000

Detailed results saved to: ./MMDocRAG/grading_results_k7.json


GPT-4o with multimodal retrieval, $k$=10

In [62]:
rag_system.config.top_k = 10

In [ ]:
results_k10 = generate_benchmark_answers_sync(
    rag_system=rag_system,
    dataset=dataset,
    output_path="./MMDocRAG/results_k10.jsonl",
    rate_limit_delay=1.0,
    resume=True
)

In [64]:
grading_data_k10 = []
for result in results_k10:
    item = {
        'id': result.get('qid', ''),
        'question': result['question'],
        'llm_response': result['answer'],               # generated answer
        'answers': result.get('reference_answers', [])  # ground truth 
    }
    grading_data_k10.append(item)

grading_results_k10 = grade_with_llm_judge(
    responses=grading_data_k10,
    client=rag_system.generator.llm,
    output_file="./MMDocRAG/grading_results_k10.json"
)

# results summary
print(f"\n{'='*50}")
print(f"Grading Summary")
print(f"{'='*50}")
print(f"Accuracy: {grading_results_k10['accuracy']:.2%}")
print(f"Correct: {grading_results_k10['correct_count']}/{grading_results_k10['total_count']}")
print(f"\nDetailed results saved to: ./MMDocRAG/grading_results_k10.json")


Grading 2000 generated responses using LLM judge...


Grading: 100%|██████████| 2000/2000 [1:07:32<00:00,  2.03s/it]


Detailed results saved to ./MMDocRAG/grading_results_k10.json

Grading Summary
Accuracy: 55.55%
Correct: 1111/2000

Detailed results saved to: ./MMDocRAG/grading_results_k10.json


### Text-only Retrieval

GPT-4o with text-only retrieval, $k$=15

In [26]:
def generate_benchmark_answers_textonly(
        rag_system: RAGSystem,
        dataset: List[Dict],
        output_path: str = "./MMDocRAG/results.jsonl",
        rate_limit_delay: float = 0.5,
        resume: bool = True
    ) -> List[Dict]:
    """
    Generates answers for all questions in the benchmark dataset.
    
    args:
    - rag_system (RAGSystem): Initialized RAGSystem instance
    - dataset (List[Dict]): a list of benchmark items
    - output_path (str): path to save results
    - rate_limit_delay (float): seconds to wait between LLM calls
    - resume (bool): whether to skip already processed items from output file

    returns:
    - a list of result dictionaries with answers
    """
    processed_questions = set()
    results = []
    
    if resume and os.path.exists(output_path):
        try:
            with open(output_path, 'r') as f:
                for line in f:
                    item = json.loads(line)
                    processed_questions.add(item.get('question', ''))
                    results.append(item)
            print(f"Resumed from {len(results)} previously processed questions\n")
        except Exception as e:
            print(f"Could not resume: {e}. Starting fresh.\n")
    
    total = len(dataset)
    
    with tqdm(total=total, desc="Generating answers", initial=len(results)) as pbar:
        for idx, item in enumerate(dataset):
            # skip if already processed
            if item.get('question') in processed_questions:
                pbar.update(1)
                continue
            
            try:
                question = item.get('question')
                qid = item.get('qid', idx)
                print(f"Question: {question}")
                
                rag_result = rag_system.query(question, use_vlm=False)
                
                result = {
                    'qid': qid,
                    'question': question,
                    'answer': rag_result.get('answer', ''),
                    'timestamp': datetime.now().isoformat(),
                    'metadata': {
                        'retrieval_mode': rag_result.get('retrieval_mode'),
                        'documents_retrieved': rag_result.get('retrieval_metadata', {}).get('documents_retrieved', 0),
                        'images_used': rag_result.get('generation_metadata', {}).get('images_used', 0),
                    }
                }
                
                if 'answer_short' in item:
                    result['reference_answer_short'] = item['answer_short']

                if 'answer_interleaved' in item:
                    result['reference_answers'] = item['answer_interleaved']
                    
                if 'text_quotes' in item:
                    result['reference_documents'] = item['text_quotes']
                
                with open(output_path, 'a') as f:
                    json.dump(result, f)
                    f.write('\n')
                
                results.append(result)
                
            except Exception as e:
                print(f"\nError at item {idx}: {str(e)}")
            
            # rate limiting
            time.sleep(rate_limit_delay)
            pbar.update(1)
    
    print(f"\nCompleted! Total processed: {len(results)}")
    
    return results

In [ ]:
rag_system.config.top_k = 15

In [ ]:
results_k15_textonly = generate_benchmark_answers_textonly(
    rag_system=rag_system,
    dataset=dataset,
    output_path="./MMDocRAG/results_k15_textonly.jsonl",
    rate_limit_delay=1.0,
    resume=True
)

In [29]:
grading_data_k15_textonly = []
for result in results_k15_textonly:
    item = {
        'id': result.get('qid', ''),
        'question': result['question'],
        'llm_response': result['answer'],               # generated answer
        'answers': result.get('reference_answers', [])  # ground truth 
    }
    grading_data_k15_textonly.append(item)

grading_results_k15_textonly = grade_with_llm_judge(
    responses=grading_data_k15_textonly,
    client=rag_system.generator.llm,
    output_file="./MMDocRAG/grading_results_k15_textonly.json"
)

# results summary
print(f"\n{'='*50}")
print(f"Grading Summary")
print(f"{'='*50}")
print(f"Accuracy: {grading_results_k15_textonly['accuracy']:.2%}")
print(f"Correct: {grading_results_k15_textonly['correct_count']}/{grading_results_k15_textonly['total_count']}")
print(f"\nDetailed results saved to: ./MMDocRAG/grading_results_k15_textonly.json")


Grading 2000 generated responses using LLM judge...


Grading: 100%|██████████| 2000/2000 [2:48:23<00:00,  5.05s/it]     


Detailed results saved to ./MMDocRAG/grading_results_k15_textonly.json

Grading Summary
Accuracy: 56.85%
Correct: 1137/2000

Detailed results saved to: ./MMDocRAG/grading_results_k15_textonly.json


GPT-4o with text-only retrieval, $k$=20

In [30]:
rag_system.config.top_k = 20

In [ ]:
results_k20_textonly = generate_benchmark_answers_textonly(
    rag_system=rag_system,
    dataset=dataset,
    output_path="./MMDocRAG/results_k20_textonly.jsonl",
    rate_limit_delay=1.0,
    resume=True
)

In [32]:
grading_data_k20_textonly = []
for result in results_k20_textonly:
    item = {
        'id': result.get('qid', ''),
        'question': result['question'],
        'llm_response': result['answer'],               # generated answer
        'answers': result.get('reference_answers', [])  # ground truth 
    }
    grading_data_k20_textonly.append(item)

grading_results_k20_textonly = grade_with_llm_judge(
    responses=grading_data_k20_textonly,
    client=rag_system.generator.llm,
    output_file="./MMDocRAG/grading_results_k20_textonly.json"
)

# results summary
print(f"\n{'='*50}")
print(f"Grading Summary")
print(f"{'='*50}")
print(f"Accuracy: {grading_results_k20_textonly['accuracy']:.2%}")
print(f"Correct: {grading_results_k20_textonly['correct_count']}/{grading_results_k20_textonly['total_count']}")
print(f"\nDetailed results saved to: ./MMDocRAG/grading_results_k20_textonly.json")


Grading 2000 generated responses using LLM judge...


Grading: 100%|██████████| 2000/2000 [1:08:08<00:00,  2.04s/it]


Detailed results saved to ./MMDocRAG/grading_results_k20_textonly.json

Grading Summary
Accuracy: 57.40%
Correct: 1148/2000

Detailed results saved to: ./MMDocRAG/grading_results_k20_textonly.json


GPT-4o with text-only retrieval, $k$=5

In [41]:
rag_system.config.top_k = 5

In [ ]:
results_k5_textonly = generate_benchmark_answers_textonly(
    rag_system=rag_system,
    dataset=dataset,
    output_path="./MMDocRAG/results_k5_textonly.jsonl",
    rate_limit_delay=1.0,
    resume=True
)

In [43]:
grading_data_k5_textonly = []
for result in results_k5_textonly:
    item = {
        'id': result.get('qid', ''),
        'question': result['question'],
        'llm_response': result['answer'],               # generated answer
        'answers': result.get('reference_answers', [])  # ground truth 
    }
    grading_data_k5_textonly.append(item)

grading_results_k5_textonly = grade_with_llm_judge(
    responses=grading_data_k5_textonly,
    client=rag_system.generator.llm,
    output_file="./MMDocRAG/grading_results_k5_textonly.json"
)

# results summary
print(f"\n{'='*50}")
print(f"Grading Summary")
print(f"{'='*50}")
print(f"Accuracy: {grading_results_k5_textonly['accuracy']:.2%}")
print(f"Correct: {grading_results_k5_textonly['correct_count']}/{grading_results_k5_textonly['total_count']}")
print(f"\nDetailed results saved to: ./MMDocRAG/grading_results_k5_textonly.json")


Grading 2000 generated responses using LLM judge...


Grading: 100%|██████████| 2000/2000 [1:08:04<00:00,  2.04s/it]


Detailed results saved to ./MMDocRAG/grading_results_k5_textonly.json

Grading Summary
Accuracy: 50.35%
Correct: 1007/2000

Detailed results saved to: ./MMDocRAG/grading_results_k5_textonly.json


GPT-4o with text-only retrieval, $k$=7


In [44]:
rag_system.config.top_k = 7

In [ ]:
results_k7_textonly = generate_benchmark_answers_textonly(
    rag_system=rag_system,
    dataset=dataset,
    output_path="./MMDocRAG/results_k7_textonly.jsonl",
    rate_limit_delay=1.0,
    resume=True
)

In [46]:
grading_data_k7_textonly = []
for result in results_k7_textonly:
    item = {
        'id': result.get('qid', ''),
        'question': result['question'],
        'llm_response': result['answer'],               # generated answer
        'answers': result.get('reference_answers', [])  # ground truth 
    }
    grading_data_k7_textonly.append(item)

grading_results_k7_textonly = grade_with_llm_judge(
    responses=grading_data_k7_textonly,
    client=rag_system.generator.llm,
    output_file="./MMDocRAG/grading_results_k7_textonly.json"
)

# results summary
print(f"\n{'='*50}")
print(f"Grading Summary")
print(f"{'='*50}")
print(f"Accuracy: {grading_results_k7_textonly['accuracy']:.2%}")
print(f"Correct: {grading_results_k7_textonly['correct_count']}/{grading_results_k7_textonly['total_count']}")
print(f"\nDetailed results saved to: ./MMDocRAG/grading_results_k7_textonly.json")


Grading 2000 generated responses using LLM judge...


Grading: 100%|██████████| 2000/2000 [1:08:00<00:00,  2.04s/it]


Detailed results saved to ./MMDocRAG/grading_results_k7_textonly.json

Grading Summary
Accuracy: 51.10%
Correct: 1022/2000

Detailed results saved to: ./MMDocRAG/grading_results_k7_textonly.json


GPT-4o with text-only retrieval, $k$=10

In [47]:
rag_system.config.top_k = 10

In [ ]:
results_k10_textonly = generate_benchmark_answers_textonly(
    rag_system=rag_system,
    dataset=dataset,
    output_path="./MMDocRAG/results_k10_textonly.jsonl",
    rate_limit_delay=1.0,
    resume=True
)

In [49]:
grading_data_k10_textonly = []
for result in results_k10_textonly:
    item = {
        'id': result.get('qid', ''),
        'question': result['question'],
        'llm_response': result['answer'],               # generated answer
        'answers': result.get('reference_answers', [])  # ground truth 
    }
    grading_data_k10_textonly.append(item)

grading_results_k10_textonly = grade_with_llm_judge(
    responses=grading_data_k10_textonly,
    client=rag_system.generator.llm,
    output_file="./MMDocRAG/grading_results_k10_textonly.json"
)

# results summary
print(f"\n{'='*50}")
print(f"Grading Summary")
print(f"{'='*50}")
print(f"Accuracy: {grading_results_k10_textonly['accuracy']:.2%}")
print(f"Correct: {grading_results_k10_textonly['correct_count']}/{grading_results_k10_textonly['total_count']}")
print(f"\nDetailed results saved to: ./MMDocRAG/grading_results_k10_textonly.json")


Grading 2000 generated responses using LLM judge...


Grading: 100%|██████████| 2000/2000 [1:07:24<00:00,  2.02s/it]


Detailed results saved to ./MMDocRAG/grading_results_k10_textonly.json

Grading Summary
Accuracy: 53.95%
Correct: 1079/2000

Detailed results saved to: ./MMDocRAG/grading_results_k10_textonly.json


GPT-4o, zero-shot

In [37]:
from models.llm_openai_azure import LLM_OpenAI_Azure

config = RAGConfig()
gpt4o_client = LLM_OpenAI_Azure(config)

In [38]:
def generate_benchmark_answers_zeroshot(
        client: LLM_OpenAI_Azure,
        dataset: List[Dict],
        output_path: str = "./MMDocRAG/results.jsonl",
        rate_limit_delay: float = 0.5,
        resume: bool = True
    ) -> List[Dict]:
    """
    Generates answers for all questions in the benchmark dataset.
    
    args:
    - client (LLM_OpenAI_Azure): Initialized LLM client
    - dataset (List[Dict]): a list of benchmark items
    - output_path (str): path to save results
    - rate_limit_delay (float): seconds to wait between LLM calls
    - resume (bool): whether to skip already processed items from output file

    returns:
    - a list of result dictionaries with answers
    """
    processed_questions = set()
    results = []
    
    if resume and os.path.exists(output_path):
        try:
            with open(output_path, 'r') as f:
                for line in f:
                    item = json.loads(line)
                    processed_questions.add(item.get('question', ''))
                    results.append(item)
            print(f"Resumed from {len(results)} previously processed questions\n")
        except Exception as e:
            print(f"Could not resume: {e}. Starting fresh.\n")
    
    total = len(dataset)
    
    with tqdm(total=total, desc="Generating answers", initial=len(results)) as pbar:
        for idx, item in enumerate(dataset):
            # skip if already processed
            if item.get('question') in processed_questions:
                pbar.update(1)
                continue
            
            try:
                question = item.get('question')
                qid = item.get('qid', idx)
                print(f"Question: {question}")
                
                rag_result = client.generate_response(question)
                
                result = {
                    'qid': qid,
                    'question': question,
                    'answer': rag_result.get('answer', ''),
                    'timestamp': datetime.now().isoformat(),
                    'metadata': {
                        'retrieval_mode': "zero_shot",
                    }
                }
                
                if 'answer_short' in item:
                    result['reference_answer_short'] = item['answer_short']

                if 'answer_interleaved' in item:
                    result['reference_answers'] = item['answer_interleaved']
                
                with open(output_path, 'a') as f:
                    json.dump(result, f)
                    f.write('\n')
                
                results.append(result)
                
            except Exception as e:
                print(f"\nError at item {idx}: {str(e)}")
            
            # rate limiting
            time.sleep(rate_limit_delay)
            pbar.update(1)
    
    print(f"\nCompleted! Total processed: {len(results)}")
    
    return results

In [ ]:
results_zeroshot = generate_benchmark_answers_zeroshot(
    client=gpt4o_client,
    dataset=dataset,
    output_path="./MMDocRAG/results_zeroshot.jsonl",
    rate_limit_delay=1.0,
    resume=True
)

In [40]:
grading_data_zeroshot = []
for result in results_zeroshot:
    item = {
        'id': result.get('qid', ''),
        'question': result['question'],
        'llm_response': result['answer'],               # generated answer
        'answers': result.get('reference_answers', [])  # ground truth 
    }
    grading_data_zeroshot.append(item)

grading_results_zeroshot = grade_with_llm_judge(
    responses=grading_data_zeroshot,
    client=rag_system.generator.llm,
    output_file="./MMDocRAG/grading_results_zeroshot.json"
)

# results summary
print(f"\n{'='*50}")
print(f"Grading Summary")
print(f"{'='*50}")
print(f"Accuracy: {grading_results_zeroshot['accuracy']:.2%}")
print(f"Correct: {grading_results_zeroshot['correct_count']}/{grading_results_zeroshot['total_count']}")
print(f"\nDetailed results saved to: ./MMDocRAG/grading_results_zeroshot.json")


Grading 2000 generated responses using LLM judge...


Grading: 100%|██████████| 2000/2000 [1:09:36<00:00,  2.09s/it]



Detailed results saved to ./MMDocRAG/grading_results_zeroshot.json

Grading Summary
Accuracy: 49.95%
Correct: 999/2000

Detailed results saved to: ./MMDocRAG/grading_results_zeroshot.json
